# Lab 2: Evaluation Metrics — PSNR, SSIM, and LPIPS

**Objectives:**
- Implement PSNR from scratch and verify against reference implementations
- Implement SSIM with 11×11 Gaussian window from first principles
- Use the `lpips` library for perceptual similarity measurement
- Design a controlled experiment showing when PSNR disagrees with perception


In [ ]:
!pip install -q numpy matplotlib scikit-image lpips torch torchvision
import numpy as np
import matplotlib.pyplot as plt
import torch
from scipy.ndimage import gaussian_filter
from skimage.metrics import structural_similarity as skimage_ssim
import lpips
%matplotlib inline
print('Ready.')

## Section 1: PSNR from Scratch

PSNR = 10 log₁₀(MAX² / MSE). For float images in [0,1], MAX=1.

In [ ]:
def psnr(img_ref, img_pred, max_val=1.0):
    """PSNR between two float32 images in [0, max_val], shape (H,W,C)."""
    mse = np.mean((img_ref.astype(np.float64) - img_pred.astype(np.float64)) ** 2)
    if mse == 0:
        return float('inf')
    return 10.0 * np.log10(max_val ** 2 / mse)

# Test image: smooth gradient (simulates a rendered 3DGS view)
H, W = 256, 256
xx, yy = np.meshgrid(np.linspace(0,1,W), np.linspace(0,1,H))
img_ref = np.stack([xx, yy, 0.5*xx + 0.5*yy], axis=-1).astype(np.float32)

# Verify
print(f'PSNR(identical):   {psnr(img_ref, img_ref)}')

for sigma in [0.001, 0.01, 0.05, 0.1]:
    noisy = np.clip(img_ref + np.random.randn(*img_ref.shape).astype(np.float32)*sigma, 0, 1)
    print(f'PSNR(sigma={sigma:.3f}): {psnr(img_ref, noisy):.2f} dB')

print('\nPSNR-MSE table:')
for mse in [1e-4, 1e-3, 1e-2, 1e-1]:
    print(f'  MSE={mse:.0e} -> PSNR={10*np.log10(1/mse):.1f} dB')

## Section 2: SSIM from Scratch

SSIM measures luminance, contrast, and structure similarity using local 11×11 Gaussian-weighted patches.

In [ ]:
def ssim_channel(img1, img2, sigma=1.5, K1=0.01, K2=0.03, L=1.0):
    """SSIM for single-channel float images in [0, L]."""
    C1, C2 = (K1*L)**2, (K2*L)**2
    filt = lambda x: gaussian_filter(x.astype(np.float64), sigma=sigma)
    mu1, mu2 = filt(img1), filt(img2)
    s1 = np.maximum(filt(img1**2) - mu1**2, 0)
    s2 = np.maximum(filt(img2**2) - mu2**2, 0)
    s12 = filt(img1*img2) - mu1*mu2
    num = (2*mu1*mu2 + C1) * (2*s12 + C2)
    den = (mu1**2 + mu2**2 + C1) * (s1 + s2 + C2)
    ssim_map = num / den
    return ssim_map.mean(), ssim_map

def ssim(img1, img2):
    return np.mean([ssim_channel(img1[...,c], img2[...,c])[0] for c in range(3)])

# Test
noisy_01 = np.clip(img_ref + np.random.randn(*img_ref.shape).astype(np.float32)*0.01, 0, 1)
noisy_10 = np.clip(img_ref + np.random.randn(*img_ref.shape).astype(np.float32)*0.10, 0, 1)

our_ssim = ssim(img_ref, noisy_01)
sk_ssim  = skimage_ssim(img_ref, noisy_01, data_range=1.0, channel_axis=-1)
print(f'Our SSIM  (sigma=0.01): {our_ssim:.4f}')
print(f'skimage SSIM:           {sk_ssim:.4f}')
print(f'Difference:             {abs(our_ssim - sk_ssim):.6f}')
print(f'\nOur SSIM  (sigma=0.10): {ssim(img_ref, noisy_10):.4f}')

# Visualize SSIM map
_, ssim_map = ssim_channel(img_ref[...,0], noisy_01[...,0])
fig, ax = plt.subplots(1, 3, figsize=(12, 4))
ax[0].imshow(img_ref); ax[0].set_title('Reference'); ax[0].axis('off')
ax[1].imshow(noisy_01); ax[1].set_title('Noisy (sigma=0.01)'); ax[1].axis('off')
im = ax[2].imshow(ssim_map, cmap='RdYlGn', vmin=0.9, vmax=1.0)
plt.colorbar(im, ax=ax[2])
ax[2].set_title('SSIM Map'); ax[2].axis('off')
plt.tight_layout(); plt.show()

## Section 3: LPIPS — Perceptual Similarity

In [ ]:
loss_fn = lpips.LPIPS(net='alex')
loss_fn.eval()

def compute_lpips(ref_np, pred_np):
    """HWC float32 [0,1] -> scalar LPIPS (lower = more similar)"""
    def to_t(img):
        return torch.tensor(img.transpose(2,0,1)).unsqueeze(0).float() * 2 - 1
    with torch.no_grad():
        return float(loss_fn(to_t(ref_np), to_t(pred_np)))

print(f'LPIPS (sigma=0.01): {compute_lpips(img_ref, noisy_01):.4f}')
print(f'LPIPS (sigma=0.10): {compute_lpips(img_ref, noisy_10):.4f}')
print('(lower = more perceptually similar)')

## Section 4: All Three Metrics vs. Noise Level

In [ ]:
sigmas = np.logspace(-3, -0.3, 18)
psnr_vals, ssim_vals, lpips_vals = [], [], []
for s in sigmas:
    img_n = np.clip(img_ref + np.random.randn(*img_ref.shape).astype(np.float32)*s, 0, 1)
    psnr_vals.append(psnr(img_ref, img_n))
    ssim_vals.append(ssim(img_ref, img_n))
    lpips_vals.append(compute_lpips(img_ref, img_n))

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].semilogx(sigmas, psnr_vals, 'b-o', ms=4)
axes[0].set_xlabel('Noise σ'); axes[0].set_ylabel('PSNR (dB)'); axes[0].set_title('PSNR')
axes[0].axhline(25, color='r', ls='--', label='25 dB'); axes[0].legend()
axes[1].semilogx(sigmas, ssim_vals, 'g-o', ms=4)
axes[1].set_xlabel('Noise σ'); axes[1].set_ylabel('SSIM'); axes[1].set_title('SSIM')
axes[2].semilogx(sigmas, lpips_vals, 'r-o', ms=4)
axes[2].set_xlabel('Noise σ'); axes[2].set_ylabel('LPIPS'); axes[2].set_title('LPIPS (lower=better)')
plt.tight_layout(); plt.show()

print(f'\n{"Sigma":>8} {"PSNR":>10} {"SSIM":>8} {"LPIPS":>8}')
for i in [0, 4, 9, 14, 17]:
    print(f'{sigmas[i]:>8.4f} {psnr_vals[i]:>10.2f} {ssim_vals[i]:>8.4f} {lpips_vals[i]:>8.4f}')

## Section 5: PSNR ≠ Perceptual Quality

A blurred image and a noisy image can have the same PSNR but very different perceptual quality.

In [ ]:
from scipy.ndimage import gaussian_filter as gf

# Blurred image
img_blur = np.stack([gf(img_ref[...,c], sigma=4.0) for c in range(3)], axis=-1).astype(np.float32)
img_blur = np.clip(img_blur, 0, 1)

# Noisy image with matched PSNR
target_psnr = psnr(img_ref, img_blur)
target_mse = 10**(-target_psnr/10)
noise_sigma = np.sqrt(target_mse)
img_noise_matched = np.clip(img_ref + np.random.randn(*img_ref.shape).astype(np.float32)*noise_sigma, 0, 1)

for name, img in [('Blurred (sigma=4)', img_blur), ('Noise (PSNR-matched)', img_noise_matched)]:
    p = psnr(img_ref, img)
    s = ssim(img_ref, img)
    l = compute_lpips(img_ref, img)
    print(f'{name:<22}: PSNR={p:.2f} dB  SSIM={s:.4f}  LPIPS={l:.4f}')

print('\n=> Same PSNR, but SSIM/LPIPS correctly identify blur as more damaging to structure.')

fig, ax = plt.subplots(1, 3, figsize=(13,4))
ax[0].imshow(img_ref); ax[0].set_title('Reference'); ax[0].axis('off')
ax[1].imshow(img_blur); ax[1].set_title(f'Blurred\nPSNR={psnr(img_ref,img_blur):.1f} SSIM={ssim(img_ref,img_blur):.3f}'); ax[1].axis('off')
ax[2].imshow(img_noise_matched); ax[2].set_title(f'Noise\nPSNR={psnr(img_ref,img_noise_matched):.1f} SSIM={ssim(img_ref,img_noise_matched):.3f}'); ax[2].axis('off')
plt.tight_layout(); plt.show()

## Key Takeaways

- **PSNR**: pure pixel-level MSE in log scale. Easy to compute, but blind to structural and perceptual quality.
- **SSIM**: captures luminance + contrast + structure in local patches. Better correlation with human judgment.
- **LPIPS**: trained on human judgments using deep features. Best proxy for perceived quality — but requires a GPU and pretrained weights.
- For 3DGS compression papers: report **all three** on the **same** test protocol. A 0.5 dB PSNR difference is meaningless if protocols differ.
- Typical 3DGS baseline quality: PSNR 24–32 dB, SSIM 0.80–0.92, LPIPS 0.05–0.20 depending on scene and dataset.
